## Homework

> Note: sometimes your answer doesn't match one of 
> the options exactly. That's fine. 
> Select the option that's closest to your solution.


In this homework, we will use the lead scoring dataset Bank Marketing dataset. Download it from [here](https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv).


In this dataset our desired target for classification task will be `converted` variable - has the client signed up to the platform or not. 

### Data preparation

* Check if the missing values are presented in the features.
* If there are missing values:
    * For caterogiral features, replace them with 'NA'
    * For numerical features, replace with with 0.0 


Split the data into 3 parts: train/validation/test with 60%/20%/20% distribution. Use `train_test_split` function for that with `random_state=1`

In [2]:
# !wget https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [13]:
df = pd.read_csv('../data/course_lead_scoring.csv')

In [14]:
df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


In [15]:
df.dtypes

lead_source                  object
industry                     object
number_of_courses_viewed      int64
annual_income               float64
employment_status            object
location                     object
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [16]:
df.isnull().sum()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [20]:
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns
categorical_columns = df.select_dtypes(include=['object']).columns

for col in numeric_columns:
    df[col] = df[col].fillna(0.0)
    
for col in categorical_columns:
    df[col] = df[col].fillna('NA')

In [21]:
numeric_columns

Index(['number_of_courses_viewed', 'annual_income', 'interaction_count',
       'lead_score', 'converted'],
      dtype='object')

In [22]:
categorical_columns

Index(['lead_source', 'industry', 'employment_status', 'location'], dtype='object')

In [23]:
df.isnull().sum()

lead_source                 0
industry                    0
number_of_courses_viewed    0
annual_income               0
employment_status           0
location                    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [42]:
X = df.drop('converted', axis=1)
y = df.converted

# First split: 80% train+val, 20% test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

# Second split: 60% train, 20% validation (0.25 of 80% is 20% of total)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=1 
)

print('Training set size:', len(X_train), f"({round(len(X_train) / (len(X_train) + len(X_val) + len(X_test))*100, 0)}%)")
print('Validation set size:', len(X_val), f"({round(len(X_val) / (len(X_train) + len(X_val) + len(X_test))*100, 0)}%)")
print('Test set size:', len(X_test), f"({round(len(X_test) / (len(X_train) + len(X_val) + len(X_test))*100, 0)}%)")

Training set size: 876 (60.0%)
Validation set size: 293 (20.0%)
Test set size: 293 (20.0%)


### Question 1: ROC AUC feature importance

ROC AUC could also be used to evaluate feature importance of numerical variables. 

Let's do that

* For each numerical variable, use it as score (aka prediction) and compute the AUC with the `y` variable as ground truth.
* Use the training dataset for that


If your AUC is < 0.5, invert this variable by putting "-" in front

(e.g. `-df_train['balance']`)

AUC can go below 0.5 if the variable is negatively correlated with the target variable. You can change the direction of the correlation by negating this variable - then negative correlation becomes positive.

Which numerical variable (among the following 4) has the highest AUC?

- `lead_score`
- `number_of_courses_viewed`
- `interaction_count`
- `annual_income`

In [ ]:
from sklearn.metrics import roc_auc_score

# List of variables to check
variables = ['lead_score', 'number_of_courses_viewed', 'interaction_count', 'annual_income']

# Dictionary to store AUC scores
auc_scores = {}

# Calculate AUC for each variable
for var in variables:
    # Get the score for this variable
    score = X_train[var]
    
    # Calculate AUC
    auc = roc_auc_score(y_train, score)
    
    # If AUC is less than 0.5, invert the variable
    if auc < 0.5:
        auc = roc_auc_score(y_train, -score)
        print(f"{var}: {auc:.4f} (inverted)")
    else:
        print(f"{var}: {auc:.4f}")
        
    auc_scores[var] = auc

# Find variable with highest AUC
best_var = max(auc_scores.items(), key=lambda x: x[1])
print(f"\nHighest AUC: {best_var[0]} ({best_var[1]:.4f})")